# Sliding-window decomposition for large n

The exact DP is `Θ(k_max n²)` time and at least `Θ(k_max n)` memory
(`prop:bb-complexity`). At `n ≳ 10^5` this becomes impractical.
§5b *Computational regime* proposes a sliding-window decomposition
that splits the sequence into overlapping windows, runs the exact DP
on each, and stitches the per-window outputs.

This is an **approximation** — stitched results do not inherit
`thm:dp-correctness` exactly and the boundary-event-sum identity holds
only per window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from bayesbreak import BayesBreakGaussian, SlidingWindowSegmenter

rng = np.random.default_rng(0)
# A longer signal than the exact DP would normally chew on.
n = 3000
k_true = 10
bps = sorted(rng.choice(np.arange(50, n - 50), size=k_true - 1, replace=False).tolist())
means = rng.normal(0, 1.5, size=k_true)
y = np.empty(n)
starts = [0, *bps, n]
for i in range(k_true):
    y[starts[i]:starts[i + 1]] = rng.normal(means[i], 0.4, size=starts[i + 1] - starts[i])
X = np.arange(n).reshape(-1, 1)
print('n =', n, 'true segments =', k_true)

## Fit with the sliding-window decomposition

In [ ]:
t0 = time.perf_counter()
sw = SlidingWindowSegmenter(
    BayesBreakGaussian(k_max=8),
    window_size=500,
    overlap=100,
).fit(X, y)
dt = time.perf_counter() - t0
print(f'sliding-window fit: n_windows={len(sw.windows_)}, total runtime={dt:.2f}s')
print(f'  k_hat = {sw.k_hat_}, |boundaries| = {len(sw.map_boundaries_) - 2}')
print(f'  approximate log p(y) = {sw.log_evidence_:.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(X.ravel(), y, color='grey', lw=0.3)
ax.plot(X.ravel(), sw.map_curve_, color='C0', lw=1.4, label='sliding-window MAP')
for b in sw.map_boundaries_[1:-1]:
    ax.axvline(b, color='C3', ls='--', lw=0.5, alpha=0.6)
for b in bps:
    ax.axvline(b, color='C2', ls=':', lw=0.7, alpha=0.7)
ax.set_xlabel('index'); ax.set_ylabel('y')
ax.legend(); plt.tight_layout(); plt.show()

## When to prefer the exact DP

Use the sliding-window when `n` would make `prop:bb-complexity` time or
memory prohibitive (typically `n ≳ 10^5`). For smaller `n`, the exact
DP gives you the boundary-event sum identity, the forward-backward
duality, and Corollary `cor:probability-error-conversion` exactly.